# 00 — Google ADK setup check

## Goal

Verify the local learner environment honestly, then automatically run one small real ADK agent checkpoint when GOOGLE_API_KEY is configured. A missing key leaves the live checkpoint waiting without failing the local check.

A live run shows only a redacted runtime timeline: user input, agent, allowlisted tool call, allowlisted tool result, and a final-answer acknowledgement. It never prints API keys, raw events, private reasoning, hidden instructions, unexpected tool data, personal paths, or live model text.

## Required learner configuration

Change the value in the next cell to your public GitHub username before requesting the optional checkpoint. Use the login shown in your GitHub profile URL, without the @ symbol. Do not infer it from Git remotes or local Git configuration. If the placeholder remains, the environment and live agent checks can still run, but no achievement, codename, or submission link will be generated.

In [ ]:
GITHUB_USERNAME = "your-github-username"  # Example: "octocat"

## 1. Local environment check

Run this notebook from the Search Agent Lab repository with its local virtual environment selected as the kernel. Set GITHUB_USERNAME in the dedicated learner configuration cell above. The notebook detects GOOGLE_API_KEY only from the local environment or untracked repository-root .env file and never prints its value.

In [ ]:
from collections.abc import Mapping
from importlib.metadata import version
from pathlib import Path
import os
import platform
import sys

from dotenv import load_dotenv
from google.adk.agents import LlmAgent
from google.adk.apps import App
from google.adk.events import Event
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types


def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / '.env.example').is_file() and (candidate / 'README.md').is_file():
            return candidate
    raise RuntimeError('Run this notebook from inside the Search Agent Lab repository.')


REPOSITORY_ROOT = find_repository_root(Path.cwd().resolve())
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from search_agent_lab.checkpoints import (
    WEEK_01,
    build_issue_form_url,
    expected_evidence,
    generate_codename,
    normalize_github_username,
)
from search_agent_lab.setup_check import assess_live_checkpoint

load_dotenv(REPOSITORY_ROOT / '.env', override=False)

LIVE_MODEL = 'gemini-3.5-flash'
AGENT_NAME = 'setup_agent'
PUBLIC_TOPIC = 'google-adk'
EXPECTED_EVIDENCE = expected_evidence(WEEK_01)

if sys.version_info < (3, 10):
    raise RuntimeError('Google ADK requires Python 3.10 or later.')
if not (REPOSITORY_ROOT / 'search_agent_lab').is_dir():
    raise RuntimeError('Search Agent Lab package directory was not found.')

adk_version = version('google-adk')
credential_available = bool(os.environ.get('GOOGLE_API_KEY'))
credential_status = 'detected locally' if credential_available else 'not configured'

print('Local environment check')
print(f'Python: {platform.python_version()}')
print(f'google-adk: {adk_version}')
print('checkpoint utilities: imported')
print(f'credential: {credential_status}')
print('Local environment check: PASS')

## 2. Deterministic local tool

The real agent can call this typed local function. Its accepted input and successful result are public, deterministic, and defined by the Week 1 checkpoint catalog.

In [ ]:
def lookup_lab_status(topic: str) -> dict[str, str]:
    '''Return the fixed public status used by this setup check.'''
    if topic != PUBLIC_TOPIC:
        return {
            'status': 'unsupported',
            'topic': 'redacted',
            'summary': 'Only the public lab topic is supported.',
        }
    return {
        'status': EXPECTED_EVIDENCE['status'],
        'topic': EXPECTED_EVIDENCE['topic'],
        'summary': EXPECTED_EVIDENCE['summary'],
    }

## 3. Safe event view

ADK events contain more than this setup check needs. These helpers expose only the expected tool name, public topic, exact catalog-defined result, and presence of non-thought final text. Everything unexpected remains redacted.

In [ ]:
def allowlisted_evidence(field: str, value: object) -> str:
    expected = EXPECTED_EVIDENCE.get(field)
    return expected if value == expected else 'redacted'


def redact_tool_call(call: object) -> dict[str, object]:
    if getattr(call, 'name', None) != EXPECTED_EVIDENCE['tool']:
        return {'redacted': True}
    arguments = getattr(call, 'args', None)
    topic = arguments.get('topic') if isinstance(arguments, Mapping) else None
    return {
        'tool': EXPECTED_EVIDENCE['tool'],
        'arguments': {'topic': allowlisted_evidence('topic', topic)},
    }


def redact_tool_result(response: object) -> dict[str, object]:
    if getattr(response, 'name', None) != EXPECTED_EVIDENCE['tool']:
        return {'redacted': True}
    raw_result = getattr(response, 'response', None)
    if not isinstance(raw_result, Mapping):
        return {'tool': EXPECTED_EVIDENCE['tool'], 'result': {'status': 'redacted'}}
    return {
        'tool': EXPECTED_EVIDENCE['tool'],
        'result': {
            field: allowlisted_evidence(field, raw_result.get(field))
            for field in ('status', 'topic', 'summary')
        },
    }


def has_nonthought_final_text(event: Event) -> bool:
    content = getattr(event, 'content', None)
    parts = getattr(content, 'parts', None) or []
    return any(
        getattr(part, 'text', None) and not getattr(part, 'thought', False)
        for part in parts
    )


def redacted_event_rows(event: Event) -> list[tuple[str, object]]:
    rows: list[tuple[str, object]] = []
    for call in event.get_function_calls():
        rows.append(('tool call', redact_tool_call(call)))
    for response in event.get_function_responses():
        rows.append(('tool result', redact_tool_result(response)))
    if getattr(event, 'error_code', None):
        rows.append(('runtime error', 'redacted ADK error'))
    if event.author != 'user' and event.is_final_response() and has_nonthought_final_text(event):
        rows.append(('final answer', 'Final response received (content omitted by setup check).'))
    return rows


def render_timeline(rows: list[tuple[str, object]]) -> None:
    print('Redacted runtime timeline')
    for stage, value in rows:
        print(f'{stage:13} | {value}')

## 4. Automatic live ADK checkpoint

Without GOOGLE_API_KEY, this cell prints a friendly waiting message and makes no model request. With a locally configured key, it automatically runs one real ADK agent invocation. The optional achievement, evidence-bound codename, and public Issue Form link appear only after the expected real tool call, exact allowlisted tool result, non-thought final answer, and a valid explicit GITHUB_USERNAME are all present. A provider-side 503 high-demand response is normally temporary; wait briefly and rerun this cell while keeping the local environment PASS.

In [ ]:
LIVE_INPUT = 'Use lookup_lab_status for the public topic google-adk, then give one short setup status.'


async def run_live_setup_check() -> list[tuple[str, object]]:
    live_agent = LlmAgent(
        name=AGENT_NAME,
        model=LIVE_MODEL,
        instruction=(
            'You are a setup checker. Call lookup_lab_status exactly once with '
            'topic google-adk, then return one short public sentence.'
        ),
        tools=[lookup_lab_status],
        generate_content_config=types.GenerateContentConfig(temperature=0),
    )
    app = App(name='setup_check', root_agent=live_agent)
    session_service = InMemorySessionService()
    session_id = 'setup-check'
    await session_service.create_session(
        app_name=app.name,
        user_id='learner',
        session_id=session_id,
    )
    runner = Runner(app=app, session_service=session_service)
    timeline: list[tuple[str, object]] = [
        ('user input', LIVE_INPUT),
        ('agent', AGENT_NAME),
    ]
    async for event in runner.run_async(
        user_id='learner',
        session_id=session_id,
        new_message=types.Content(role='user', parts=[types.Part(text=LIVE_INPUT)]),
    ):
        timeline.extend(redacted_event_rows(event))
    return timeline


if not credential_available:
    print('Live agent checkpoint is waiting for GOOGLE_API_KEY; no model request was made.')
else:
    try:
        live_timeline = await run_live_setup_check()
    except Exception:
        print('Live checkpoint did not complete. Verify local key, network, quota, and model access.')
    else:
        render_timeline(live_timeline)
        assessment = assess_live_checkpoint(live_timeline, WEEK_01)
        if not assessment.succeeded or assessment.evidence is None:
            print(assessment.message)
        elif GITHUB_USERNAME.strip() in {'', 'your-github-username'}:
            print('Set GITHUB_USERNAME explicitly to your public GitHub login, then rerun this cell.')
        else:
            try:
                normalized_username = normalize_github_username(GITHUB_USERNAME)
            except ValueError:
                print('GITHUB_USERNAME is invalid. Use only your public GitHub login.')
            else:
                checkpoint_codename = generate_codename(
                    normalized_username, WEEK_01, assessment.evidence
                )
                submission_url = build_issue_form_url(
                    normalized_username, WEEK_01, assessment.evidence
                )
                print(WEEK_01.achievement_message)
                print(f'Week 1 agent codename: {checkpoint_codename}')
                print(f'Optional public Issue Form: {submission_url}')
                print('Public honor-system celebration only; not authentication or formal grading.')

## Next steps

- Local environment check: PASS means the repository, Python, ADK package, checkpoint utilities, and credential detection path are ready.
- A successful live checkpoint confirms only this locally configured Gemini agent path; it does not authorize deployment or Google Cloud setup.
- The optional Issue is public and honor-system based, not authentication, certification, or formal grading.
- Keep .env untracked and never save or commit notebook outputs containing external runtime content.
- The upstream shop_agent.ipynb will be studied later from a pinned separate checkout, not copied into this repository.